# FleetGuard AI — обучение YOLO11n-seg на CarDD

**Порядок действий**

1. `Среда выполнения` → `Сменить среду выполнения` → **T4 GPU** → `Сохранить`.
2. `Среда выполнения` → **Выполнить все**.
3. Когда появится окно — разрешить доступ к Google Drive.

**Что делает ноутбук:** клонирует репозиторий, скачивает CarDD (4000 изображений, ~4 ГБ), собирает COCO-разметку из трёх файлов, готовит YOLO-seg датасет, проверяет его, обучает 100 эпох с early stopping (patience 20), считает метрики на val и складывает артефакты на Drive.

**Сколько ждать:** 10–20 минут на данные, 1–3 часа на обучение. Сессию не закрывать; чекпоинты пишутся на Drive.

**В конце** ноутбук напечатает `metrics.md`, `sha256` и путь к `best.pt`. Эти три вещи нужно прислать в чат.

In [ ]:
import subprocess
import sys

print("python:", sys.version.split()[0])
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
).stdout.strip()
print("gpu:", gpu or "не найден — включите T4: Среда выполнения → Сменить среду выполнения")

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/fleetguard-ai")
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/minbaevv/fleetguard-ai.git", str(REPO)],
        check=True,
    )
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3,<9.0", "huggingface_hub"], check=True)
print("repo:", REPO)

In [ ]:
from pathlib import Path

RUNS = Path("/content/drive/MyDrive/fleetguard-cv/runs")
try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception as error:
    RUNS = Path("/content/runs")
    print("Drive не подключён, результаты останутся в сессии:", error)
RUNS.mkdir(parents=True, exist_ok=True)
print("runs:", RUNS)

In [ ]:
import os
from pathlib import Path

from huggingface_hub import snapshot_download

DATA = Path("/content/fleetguard-ai/datasets/cardd")
snapshot_download(
    repo_id="shawnmichael/CarDD",
    repo_type="dataset",
    local_dir=str(DATA),
    allow_patterns=[
        "images/train/*.jpg",
        "images/val/*.jpg",
        "images/test/*.jpg",
        "annotations/instances_*.json",
    ],
    max_workers=8,
)

images = DATA / "images_all"
images.mkdir(parents=True, exist_ok=True)
for split in ("train", "val", "test"):
    for image in (DATA / "images" / split).glob("*.jpg"):
        target = images / image.name
        if not target.exists():
            os.link(image, target)
print("images:", len(list(images.glob("*.jpg"))))

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/fleetguard-ai")
DATA = REPO / "datasets" / "cardd"
annotations = [
    str(DATA / "annotations" / f"instances_{split}2017.json")
    for split in ("train", "val", "test")
]
merged = DATA / "instances_all.json"

subprocess.run(
    [sys.executable, "cv/training/merge_coco_splits.py", "--annotations", *annotations, "--out", str(merged)],
    cwd=REPO,
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "cv/training/prepare_dataset.py",
        "--images",
        str(DATA / "images_all"),
        "--annotations",
        str(merged),
        "--out",
        str(REPO / "datasets" / "cardd-yolo"),
        "--seed",
        "42",
        "--force",
    ],
    cwd=REPO,
    check=True,
)
subprocess.run(
    [sys.executable, "cv/training/train.py", "--data", str(REPO / "datasets" / "cardd-yolo"), "--dry-run"],
    cwd=REPO,
    check=True,
)

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/fleetguard-ai")
RUNS = Path("/content/drive/MyDrive/fleetguard-cv/runs")
if not RUNS.exists():
    RUNS = Path("/content/runs")
RUNS.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        sys.executable,
        "cv/training/train.py",
        "--data",
        str(REPO / "datasets" / "cardd-yolo"),
        "--device",
        "0",
        "--project",
        str(RUNS),
        "--name",
        "yolo11n-seg-cardd",
    ],
    cwd=REPO,
    check=True,
)

In [ ]:
import hashlib
from pathlib import Path

RUNS = Path("/content/drive/MyDrive/fleetguard-cv/runs")
if not RUNS.exists():
    RUNS = Path("/content/runs")
run_dir = RUNS / "yolo11n-seg-cardd"
weights = run_dir / "weights" / "best.pt"
metrics = run_dir / "metrics.md"

print("run_dir:", run_dir)
if weights.exists():
    print("best.pt:", weights)
    print("size_mb:", round(weights.stat().st_size / 1024 / 1024, 1))
    print("sha256:", hashlib.sha256(weights.read_bytes()).hexdigest())
else:
    print("best.pt не найден")
print()
print(metrics.read_text() if metrics.exists() else "metrics.md не найден")

## Что прислать в чат

1. Содержимое `metrics.md` — его печатает предыдущая ячейка.
2. Строку `sha256`.
3. Ссылку на `best.pt`: Google Drive → `fleetguard-cv/runs/yolo11n-seg-cardd/weights/best.pt` → ПКМ → `Поделиться` → `Все, у кого есть ссылка` → скопировать ссылку.

Веса в git не коммитятся. В README публикуются только измеренные значения.